# Fisher forecast for anisotropic terms in diagonal part of covariance matrix

Fisher forecast from the BipoSH off-diagonal correction $\delta C_{\ell,m}$, for the mixed
CDM+VFDM pairs $(f, \log_{10} m)$ on the 95% contour.

$\left.\left\langle a_{\ell m} a_{\ell' m'}^{*} \right\rangle\right|_{\mathrm{diag}} = C_\ell \, \delta_{\ell\ell'} \delta_{m m'} + g \, \delta C_{\ell m,\ell m}$ 

In [1]:
from classy import Class
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
from scipy.interpolate import interp1d
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.family'] = 'serif'

## Wigner 3j symbols

In [2]:
from scipy.special import gammaln

# Abramowitz & Stegun 27.9.1:
# gammaln(x) = ln Γ(x) and Γ(n+1) = n!, so every factorial n! in the A&S formula
# appears below as gammaln(n + 1). E.g. (j1+j2-j)! -> gammaln(j1+j2-j+1), and
# 1/(j1+j2+j+1)! -> -gammaln(j1+j2+j+2). The expression is A&S 27.9.1 verbatim,
# just written in log space (the overall 0.5* implements the square roots).

def clebsch_gordan(j1, m1, j2, m2, j, m):
    """<j1 m1 j2 m2 | j m> — A&S 27.9.1 (Condon–Shortley phase).
    Works for integer or half-integer arguments. gammaln keeps it stable to large j."""
    if abs(m1 + m2 - m) > 1e-9:                                  # δ(m, m1+m2)
        return 0.0
    if not (abs(j1 - j2) <= j <= j1 + j2):                       # triangle rule
        return 0.0
    if abs(m1) > j1 or abs(m2) > j2 or abs(m) > j:
        return 0.0

    # line 1 (sqrt prefactor) · line 2 (sqrt of the m-factorials), all in log space
    logpre = 0.5 * (
        gammaln(j1 + j2 - j + 1) + gammaln(j + j1 - j2 + 1)
        + gammaln(j + j2 - j1 + 1) + np.log(2 * j + 1) - gammaln(j1 + j2 + j + 2)
        + gammaln(j1 + m1 + 1) + gammaln(j1 - m1 + 1)
        + gammaln(j2 + m2 + 1) + gammaln(j2 - m2 + 1)
        + gammaln(j + m + 1)  + gammaln(j - m + 1)
    )

    # sum over k: every factorial argument must stay >= 0
    kmin = int(round(max(0, j2 - j - m1, j1 - j + m2)))
    kmax = int(round(min(j1 + j2 - j, j1 - m1, j2 + m2)))
    if kmax < kmin:
        return 0.0
    ks = np.arange(kmin, kmax + 1)
    logt = -(gammaln(ks + 1) + gammaln(j1 + j2 - j - ks + 1)
             + gammaln(j1 - m1 - ks + 1) + gammaln(j2 + m2 - ks + 1)
             + gammaln(j - j2 + m1 + ks + 1) + gammaln(j - j1 - m2 + ks + 1))
    signs = np.where(ks % 2 == 0, 1.0, -1.0)
    lmax = logt.max()                            # factor out largest term (cancellation-safe)
    return float(np.exp(logpre + lmax) * np.sum(signs * np.exp(logt - lmax)))

def wigner_3j(j1, j2, j3, m1, m2, m3):
    """(j1 j2 j3; m1 m2 m3) = (-1)^(j1-j2-m3)/√(2j3+1) · <j1 m1 j2 m2 | j3 -m3>."""
    if abs(m1 + m2 + m3) > 1e-9:                  # 3j requires m1+m2+m3 = 0
        return 0.0
    cg = clebsch_gordan(j1, m1, j2, m2, j3, -m3)
    if cg == 0.0:
        return 0.0
    return (-1.0) ** int(round(j1 - j2 - m3)) / np.sqrt(2 * j3 + 1) * cg

## CLASS runs

In [3]:
pairs = [(0.01, -25.515), (0.10,  -25.15),  (0.50,  -24.578), (1.00,  -24.436)]

vf_settings = {'N_ncdm' : 0.,
            'Omega_k' : 0.,
            'Omega_fld' : 0.,
            'Omega_dcdmdr' : 0.0,
            'YHe' : 'BBN',
            'gauge' : 'synchronous',
            'P_k_max_h/Mpc' : 100.,
            'l_max_scalars': 2500,
            'P_k_ini type' : 'analytic_Pk',
            'z_pk': '0,20,100',
            'modes' : 's',
            'recombination' : 'RECFAST',
            'output' : 'tCl,pCl,lCl, mPk, mTk',
            'lensing': 'yes',
            'attractor_ic_vf' : 'yes',
            'vector_background_mode': 'bianchi',
            'svt_coupling': 'no',
            'format': 'camb',
            'ic' : 'ad',}

Omega_cdm = 0.2548  # Omega_vf (bestfit) + Omega_cdm (bestfit)
As = 2.102547e-09 #best fit mixed
ns = 9.687243e-01 #best fit mixed
k_pivot = 0.05

transfer_tables = {}   # (frac, logm) -> {'T0': Theta_l(k) at gamma=0, 'Tpi2': at gamma=pi/2}

for pi_idx, (frac_p, logm_p) in enumerate(pairs):
    tables = {}
    for key, gamma in [('T0', 0), ('Tpi2', np.pi/2)]:
        print(f'Running CLASS: f={frac_p}, log10(m)={logm_p}, gamma_Ak={key[1:] or "0"} ...')
        c = Class()
        c.set(vf_settings)
        c.set({
            'Omega_cdm'    : Omega_cdm * (1 - frac_p),
            'Omega_vf'     : frac_p * Omega_cdm,
            'gamma_Ak'     : gamma,
            'ln10^{10}m_a' : logm_p,
        })
        c.compute()
        tr = c.get_cmb_transfer()
        tables[key] = tr['T']
        if pi_idx == 0 and key == 'T0':
            k_mixto = tr['k']
            l_mixto = tr['l'].astype(int)
        c.struct_cleanup()
        c.empty()
    transfer_tables[(frac_p, logm_p)] = tables

Running CLASS: f=0.01, log10(m)=-25.515, gamma_Ak=0 ...
Running CLASS: f=0.01, log10(m)=-25.515, gamma_Ak=pi2 ...
Running CLASS: f=0.1, log10(m)=-25.15, gamma_Ak=0 ...
Running CLASS: f=0.1, log10(m)=-25.15, gamma_Ak=pi2 ...
Running CLASS: f=0.5, log10(m)=-24.578, gamma_Ak=0 ...
Running CLASS: f=0.5, log10(m)=-24.578, gamma_Ak=pi2 ...
Running CLASS: f=1.0, log10(m)=-24.436, gamma_Ak=0 ...
Running CLASS: f=1.0, log10(m)=-24.436, gamma_Ak=pi2 ...


## Fisher

In [4]:
# integrals in deltaCl 
def precompute_integrals(k, T0, Tpi2, P_R):
    n_l  = T0.shape[0] 
    dT   = Tpi2 - T0  
    Cl = np.array([4*np.pi * CubicSpline(k, P_R * T0[i,:]**2 / k).integrate(k[0], k[-1]) for i in range(n_l)])
    int_dT2  = np.array([4*np.pi * CubicSpline(k, P_R * dT[i, :]**2 / k).integrate(k[0], k[-1]) for i in range(n_l)])
    int_T0dT = np.array([4*np.pi * CubicSpline(k, P_R * T0[i, :] * dT[i, :] / k).integrate(k[0], k[-1]) for i in range(n_l)])

    return Cl, int_dT2, int_T0dT



def cross_integral(k, dT_l, dT_lp, P_R):

    return 4*np.pi * CubicSpline(k, P_R * dT_l * dT_lp / k).integrate(k[0], k[-1])



def compute_fisher_Cl(k, l_arr, T0, Tpi2):

    P_R = As * (k / k_pivot)**(ns - 1.)
    
    C_l, int_dT2, int_T0dT = precompute_integrals(k, T0, Tpi2, P_R)

    w2_diag = {}
    w4_diag = {}

    for l in l_arr:
        m_vals = np.arange(-l, l + 1)
        for m in m_vals:
            w2_diag[(l,m)] = (-1)**int(m)*wigner_3j(l, 2, l, m, 0, -m)
            w4_diag[(l,m)] = (-1)**int(m)*wigner_3j(l, 4, l, m, 0, -m)

    F = 0.0

    for idx_l, l in enumerate(l_arr):

        if C_l[idx_l] == 0:
            continue

        int_cross = int_dT2[idx_l]
        int_sym   = 2.0 * int_T0dT[idx_l]

        m_vals = np.arange(-l, l + 1)

        w2_arr = np.array([w2_diag[(l,m)] for m in m_vals])
        w4_arr = np.array([w4_diag[(l,m)] for m in m_vals])

        I1_arr = (8*np.sqrt(9*(2*l+1)**2)*w4_arr * wigner_3j(l, 4, l, 0, 0, 0)
                  +20*np.sqrt(5*(2*l+1)**2)*w2_arr * wigner_3j(l, 2, l, 0, 0, 0)
                  +7)
        I2_arr = (2*np.sqrt(5*(2*l+1)**2)*w2_arr * wigner_3j(l, 2, l, 0, 0, 0) +1)

        # delta C_{lm,lm}
        dC_lm = ((1/35)*I1_arr*int_cross+(1/3)*I2_arr*int_sym)
        # delta C_l = average over m
        deltaCl = np.mean(dC_lm)

        F += ((2*l+1)* deltaCl**2 / C_l[idx_l]**2)

    return 0.5 * F

In [5]:
for frac_p, logm_p in pairs:

    T0_p   = transfer_tables[(frac_p, logm_p)]['T0']
    Tpi2_p = transfer_tables[(frac_p, logm_p)]['Tpi2']

    F_Cl = compute_fisher_Cl(k_mixto, l_mixto, T0_p, Tpi2_p)

    print(f"f={frac_p}, log(m)={logm_p}")
    print(f"F_gg     = {F_Cl:.6e}")
    print(f"sigma_g  = {1/np.sqrt(F_Cl):.4f}\n")

f=0.01, log(m)=-25.515
F_gg     = 1.729572e-02
sigma_g  = 7.6038

f=0.1, log(m)=-25.15
F_gg     = 9.172805e-02
sigma_g  = 3.3018

f=0.5, log(m)=-24.578
F_gg     = 4.994496e-01
sigma_g  = 1.4150

f=1.0, log(m)=-24.436
F_gg     = 8.374288e-01
sigma_g  = 1.0928

